# Update Experiment Config Dict with replaced Algorithm Names:

In [23]:
import run_experiment
import experiment_config as exc

In [62]:
config = exc.import_config(fn="/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/config.json")
alg = "NetVlad"
config

{'data_final_root': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data',
 'global_descriptor_model': 'MixVPR',
 'local_feature_model': 'superpoint+lightglue',
 'places': {'Mahidol_University': {'ICT': ['1.1_MixVPR']}},
 'path_to_images': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_MixVPR/perspectives',
 'ground_truth_img_list': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_MixVPR/groundtruth_img_dataset_ICT_1.1_MixVPR_10P.csv',
 'img_list_attr': 'image_name',
 'img_ext': 'png',
 'exp_type': 'localize',
 'results_out_path': 'Config__v2-1_c',
 'root_dir': '/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/',
 'results_fn': 'Results_localize.xlsx',
 'num_trials': 5,
 'exp_version': 'v2-1_c',
 'exp_description': '\n            v2-1_c - Mapping on 1.1 and localization on the 1.1 dataset\n            v2-1_b - Change to UNav-V2 - Image localization experiment - 5 VPR algorithms.

In [57]:
def update_dict(algorithm:str, config_exp:dict, place='Mahidol_University', building='ICT') -> dict:
    """
    Will replace `global_descriptor_model:str` with `algorithm:str`.
    Will break if a new dictionary is added.
    """
    alg_old = config_exp['global_descriptor_model']
    for k,v in config_exp.items():
        if isinstance(v, str):
            config_exp[k] = v.replace(alg_old,alg)
        elif isinstance(v, dict):
            l = config_exp[k][place][building]
            for i,f in enumerate(l):
                l[i] = l[i].replace(alg_old,alg)
            config_exp[k][place][building] = l
        elif isinstance(v, int):
            pass
        else:
            raise ValueError(f'Unsupported value type {type(v)}')
    return config_exp

In [64]:
update_dict(alg, config)

{'data_final_root': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data',
 'global_descriptor_model': 'NetVlad',
 'local_feature_model': 'superpoint+lightglue',
 'places': {'Mahidol_University': {'ICT': ['1.1_NetVlad']}},
 'path_to_images': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_NetVlad/perspectives',
 'ground_truth_img_list': '/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_NetVlad/groundtruth_img_dataset_ICT_1.1_NetVlad_10P.csv',
 'img_list_attr': 'image_name',
 'img_ext': 'png',
 'exp_type': 'localize',
 'results_out_path': 'Config__v2-1_c',
 'root_dir': '/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/',
 'results_fn': 'Results_localize.xlsx',
 'num_trials': 5,
 'exp_version': 'v2-1_c',
 'exp_description': '\n            v2-1_c - Mapping on 1.1 and localization on the 1.1 dataset\n            v2-1_b - Change to UNav-V2 - Image localization experiment - 5 VPR algori

# Distance Errors with Trial Nums:

In [59]:
from distance_error import save_calculated_distance_error
import experiment_config

import pandas as pd
import cv2
import os
import json
import numpy as np
import sys
from scipy.spatial import distance
from os.path import dirname,join,exists,realpath





def save_calculated_distance_error(exp_conf, prediction_filename):
    pixel_to_meter_ratio = 27.14
    dataset_name = 'TestSet_360_v1.1_a'
    algorithm = exp_conf['global_descriptor_model']
    root_dir = exp_conf['root_dir']
    
    est_df = pd.read_excel(os.path.join(root_dir, exp_conf['results_out_path'], prediction_filename))
    est_df['image_name'] = est_df['image_fn'].apply(lambda s : s.split('/')[-1].split('.')[0])
    est_df = est_df.drop(columns=['error'],axis=1)
    ground_truth = pd.read_csv(exp_conf['ground_truth_img_list'])
    
    eu_dist_error_df = est_df.set_index('image_name').dropna().join(ground_truth.set_index('image_name'), how='inner', lsuffix='_est', rsuffix='_ground').apply(
        lambda row : distance.euclidean([row['coordx'], row['coordy']], [row['cx'], row['cy']]), axis=1).to_frame()

    eu_dist_error_df['time_ms'] = est_df.set_index('image_name').dropna()['time_ms']
    eu_dist_error_df['trial_num'] = est_df.set_index('image_name').dropna()['trial_num']
    eu_dist_error_df = eu_dist_error_df.rename(columns={0:'error_distance_pixel'})
    eu_dist_error_df['error_distance_meter'] = eu_dist_error_df['error_distance_pixel'] / pixel_to_meter_ratio
    eu_dist_error_df = eu_dist_error_df[['error_distance_pixel','error_distance_meter','time_ms','trial_num']]
    
    out_path = exp_conf['results_out_path']
    distance_error_fn = f"{root_dir}/{out_path}/Results_distance_error_{algorithm}.xlsx"
    eu_dist_error_df.to_excel(distance_error_fn) # Save errors to Excel
    print(f"==== Distance errors were saved to {distance_error_fn} ====")
    means = eu_dist_error_df[['error_distance_meter','trial_num']].groupby('trial_num').mean()['error_distance_meter'].tolist()
    stds = eu_dist_error_df[['error_distance_meter','trial_num']].groupby('trial_num').std()['error_distance_meter'].tolist()

    means_time_ms = (eu_dist_error_df[['time_ms','trial_num']].groupby('trial_num').mean()['time_ms'] / (len(est_df)/exp_conf['num_trials'])).tolist()
    stds_time_ms = (eu_dist_error_df[['time_ms','trial_num']].groupby('trial_num').std()['time_ms'] / (len(est_df)/exp_conf['num_trials'])).tolist()

    r = {'exp_version' : exp_conf['exp_version'],      
            'distance_error_meters_mean_mean' : np.nanmean(means),
            'distance_error_meters_std_mean' : np.nanmean(stds),
            'distance_error_meters_per_trial_mean' : means,
            'distance_error_meters_per_trial_std' : stds,      
            'time_localization_mean_mean' : np.nanmean(means_time_ms),
            'time_localization_std_mean' : np.nanmean(stds_time_ms),
            'time_localization_mean_per_trial' : means_time_ms,
            'time_localization_std_per_trial' : stds_time_ms,
            'num_images_localized' : len(eu_dist_error_df),
            'num_images_total' : len(est_df),
            'perc_images_localized' : (len(eu_dist_error_df)/exp_conf['num_trials']) / (len(est_df)/exp_conf['num_trials']),
            'num_images_per_trial' : len(est_df)/exp_conf['num_trials'],
            'num_trials': exp_conf['num_trials'],
            'dataset_name' : dataset_name,
            'algorithm' : algorithm
        }

    with open(f"{root_dir}/{out_path}/distance_error_meta_{algorithm}.json", 'w') as f:
        f.write(json.dumps(r))

    # return distance_error_fn
    return distance_error_fn, r


In [70]:
dft = pd.read_excel('/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/Results_localize_CricaVPR.xlsx')
dft = pd.read_excel('/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/Results_distance_error_CricaVPR.xlsx')
dft
# dft[dft['trial_num']==1].drop(columns=['error'],axis=1).dropna()

,image_name,error_distance_pixel,error_distance_meter,time_ms,trial_num
0,000018_pitch00_yaw13,16.241218,0.598424,4077,0
1,000074_pitch00_yaw15,0.358412,0.013206,27898,0
2,000123_pitch00_yaw15,1.572982,0.057958,29979,0
3,000267_pitch00_yaw14,0.598472,0.022051,36448,0
4,000082_pitch00_yaw02,4.601234,0.169537,23216,0
...,...,...,...,...,...
493,000218_pitch00_yaw08,1.750201,0.064488,4697,2
494,000071_pitch00_yaw08,0.729227,0.026869,9008,2
495,000230_pitch00_yaw05,0.501334,0.018472,13547,2
496,000207_pitch00_yaw01,0.401584,0.014797,5581,2


In [68]:
# from distance_error import save_calculated_distance_error
import experiment_config

for alg in ['MixVPR',
            'CricaVPR',
            'NetVlad',
            'DinoV2Salad',
            'AnyLoc'
           ]:
    exp_conf = experiment_config.import_config(fn="/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/config.json")
    prediction_filename = f"Results_localize_{alg}.xlsx"
    exp_conf = update_dict(alg, exp_conf)
    
    print( save_calculated_distance_error(exp_conf, prediction_filename) )
    print('\n\n\n\n\n\n\n')

==== Distance errors were saved to /home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs//Config__v2-1_c/Results_distance_error_MixVPR.xlsx ====
('/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs//Config__v2-1_c/Results_distance_error_MixVPR.xlsx', {'exp_version': 'v2-1_c', 'distance_error_meters_mean_mean': 0.19248985666171967, 'distance_error_meters_std_mean': 0.24097544801074422, 'distance_error_meters_per_trial_mean': [0.20807083763064285, 0.1838857443140269, 0.18551298804048924], 'distance_error_meters_per_trial_std': [0.25131178486101563, 0.2248620849887474, 0.24675247418246965], 'time_localization_mean_mean': 11.333349468559001, 'time_localization_std_mean': 6.367792479165676, 'time_localization_mean_per_trial': [11.459344973012591, 10.998547315104066, 11.542156117560346], 'time_localization_std_per_trial': [7.705369781628511, 5.395301521741366, 6.00270613412715], 'num_images_localized': 813, 'num_images_total': 1809, 'perc_images_localized': 0.449419568

In [51]:
# eu_dist_error_df['time_ms'] = est_df.set_index('image_name').dropna()['time_ms']
# eu_dist_error_df

# eu_dist_error_df['trial_num'] = est_df.set_index('image_name').dropna()['trial_num']
# eu_dist_error_df = eu_dist_error_df.rename(columns={0:'error_distance_pixel'})
# eu_dist_error_df['error_distance_meter'] = eu_dist_error_df['error_distance_pixel'] / pixel_to_meter_ratio
# eu_dist_error_df = eu_dist_error_df[['error_distance_pixel','error_distance_meter','time_ms','trial_num']]

# Debug a Trial Experiment

In [ ]:
from run_experiment import *
import experiment_config

In [ ]:
config = exc.import_config(fn="/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/config.json")
alg = "NetVlad"
config

In [ ]:
/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000268_pitch00_yaw02.png

In [ ]:
# import argparse
# parser = argparse.ArgumentParser()
# parser.add_argument('-a', '--algorithm', type=str, default=, )
# parser.add_argument('-x', '--experiment', type=str, default=, help='Experiment config filepath for experimental VPR localization.')
# args = parser.parse_args()
class A:
    experiment = "/home/nattachart.tak/PhD/Trial_New_UNav/UNav/experiment/configs/Config__v2-1_c/config.json"
    algorithm = "CricaVPR"
args = A()
config_exp = experiment_config.import_config(fn=args.experiment)
config_exp = update_dict(args.algorithm, config_exp)

DATA_FINAL_ROOT = config_exp.get('data_final_root', "/mnt/data/UNav-IO/data")
FEATURE_MODEL = args.algorithm #onfig_exp.get('global_descriptor_model', "DinoV2Salad")
config_exp['global_descriptor_model'] = args.algorithm
LOCAL_FEATURE_MODEL = config_exp.get('local_feature_model', "superpoint+lightglue")
PLACES = config_exp.get('places')
# , {
#                 "New_York_City": {
#                     "LightHouse": ["3_floor", "4_floor", "6_floor"]
#                 }
#             }

config = UNavConfig(
    data_final_root=DATA_FINAL_ROOT,
    places=PLACES,
    global_descriptor_model=FEATURE_MODEL,
    local_feature_model=LOCAL_FEATURE_MODEL
)
localizor_config = config.localizer_config
localizer = UNavLocalizer(localizor_config)
localizer.load_maps_and_features()


r = []
groundtruth_df = pd.read_csv(config_exp['ground_truth_img_list'])
image_filepaths = groundtruth_df[config_exp['img_list_attr']].apply(lambda x : f"{config_exp['path_to_images']}/{x}.{config_exp['img_ext']}").tolist()
config_exp['num_trials'] = 1000
#image_filepaths = ["/mnt/data/UNav-IO/test/photos/LightHouse/3-1.jpg"]
for trial_num in range(config_exp['num_trials']):
    for image_filepath in image_filepaths:
        if image_filepath not in ["/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000087_pitch00_yaw17.png",
                                 "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000268_pitch00_yaw02.png",
                                 "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000051_pitch00_yaw15.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000146_pitch00_yaw07.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000281_pitch00_yaw12.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000143_pitch00_yaw07.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000023_pitch00_yaw17.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000043_pitch00_yaw08.png",
                                    "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000224_pitch00_yaw02.png"
                                ]:
        # if image_filepath != "/home/nattachart.tak/Data/experiments/Mapping/data/unav2-data/Mahidol_University/ICT/1.1_CricaVPR/perspectives/000268_pitch00_yaw02.png":
            continue
        img = get_image(image_filepath)
        cx,cy,ang = None,None,None
        error = None
        time_start = ExperimentTime.get_time_ms()
        # try:
        global_feat, local_feat_dict = localizer.extract_query_features(img)
        top_candidates = localizer.vpr_retrieve(global_feat, top_k=50)
        candidates_data = localizer.get_candidates_data(top_candidates)
        #print(candidates_data)
        best_map_key, pnp_pairs = get_best_map_key(localizer, local_feat_dict, candidates_data)
        refinement_queue = {best_map_key: {"pairs": [], "initial_poses": [], "pps": []}}
        time_start_multiframe = ExperimentTime.get_time_ms()
        refine_result = localizer.multi_frame_pose_refine(
            pnp_pairs, img.shape, refinement_queue[best_map_key]
        )
        print(f'Multiframe time (ms): {ExperimentTime.get_time_ms_duration(time_start_multiframe)}')

        cx,cy,ang = get_pose(localizer, best_map_key, refine_result)
        # except Exception as e:
        #     error = str(e)
        total_time = ExperimentTime.get_time_ms_duration(time_start)
        r += [
            {'coordx':cx,
                'coordy':cy,
                'angle':ang,
                'time_ms':total_time,
                'image_fn':image_filepath,
                'trial_num':trial_num,
                'error':error,
            }]

root_dir = config_exp['root_dir']
filename_parts = config_exp['results_fn'].split('.')
results_fn = f'{".".join(filename_parts[:-1])}_{args.algorithm}.{filename_parts[-1]}'

out_path = config_exp['results_out_path']
result_file = f'{root_dir}/{out_path}/{results_fn}'
pd.DataFrame.from_records(r).to_excel(result_file)
print(f"==== Predicted coordinates were saved to '{result_file}'. ====")

print("==== Calculate distance errors ====")
distance_error_fn = distance_error.save_calculated_distance_error(config_exp, results_fn)
print(f"==== Distance errors were saved to {distance_error_fn} ====")